# Keyword Expansion from Page

Use this notebook to start with an existing high-traffic page and identify related keyword demand that should either be folded into the page or spun out into a new article.

### Quick start (beginner-friendly)
1. Run the **Setup (run once)** cell.
2. In **Parameters**, set `PAGE_PATH` and review the market defaults.
3. Run the remaining cells from top to bottom.
4. If Google Ads credentials are not ready yet, the notebook will still return the page's existing GSC query drivers.

### Google Ads setup
Set either `GOOGLE_ADS_CONFIGURATION_FILE` or these environment variables before running keyword-idea cells:
- `GOOGLE_ADS_CUSTOMER_ID`
- `GOOGLE_ADS_DEVELOPER_TOKEN`
- `GOOGLE_ADS_CLIENT_ID`
- `GOOGLE_ADS_CLIENT_SECRET`
- `GOOGLE_ADS_REFRESH_TOKEN`
- optional `GOOGLE_ADS_LOGIN_CUSTOMER_ID`

### Links
- GitHub repo: [github.com/aidanm-lla/lla-data](https://github.com/aidanm-lla/lla-data)
- Open this notebook in Colab: [Keyword Expansion from Page](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/14_keyword_expansion_from_page.ipynb)

### Other notebooks
- [Query Drivers by Page](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/03_query_drivers_by_page.ipynb)
- [Opportunity Watchlist](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/04_opportunity_watchlist.ipynb)
- [Top Search Queries (Sitewide)](https://colab.research.google.com/github/aidanm-lla/lla-data/blob/main/notebooks/07_top_search_queries_sitewide.ipynb)


In [ ]:
#@title Setup (run once)
import os
import sys
from urllib.parse import urljoin

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    if not os.path.exists("lla-data"):
        !git clone -q https://github.com/aidanm-lla/lla-data.git
    repo = os.path.abspath("lla-data")
    if repo not in sys.path:
        sys.path.insert(0, repo)
    !pip install -U -q "plotly>=6.1.1" "kaleido>=1.2.0" "google-ads>=29.2.0"
else:
    for p in ("..", "../.."):
        ap = os.path.abspath(p)
        if ap not in sys.path:
            sys.path.insert(0, ap)

import pandas as pd
import plotly.express as px
from IPython.display import display
from google.cloud import bigquery

import lifeline_theme
from lla_data import config
from lla_data.bq import build_date_params, default_query_window, get_client, load_sql_template, run_query
from lla_data.google_ads_keywords import (
    GoogleAdsConfigurationError,
    annotate_page_opportunities,
    generate_keyword_ideas,
)

lifeline_theme.inject_fonts()
client = get_client()


In [ ]:
#@title Parameters
DAYS_BACK = 90 #@param {type:"integer"}
PAGE_PATH = "/" #@param {type:"string"}
COUNTRY = "Australia" #@param {type:"string"}
LANGUAGE = "English" #@param {type:"string"}
TOP_EXISTING_QUERIES = 20 #@param {type:"integer"}
MAX_IDEAS = 100 #@param {type:"integer"}
MIN_AVG_MONTHLY_SEARCHES = 20 #@param {type:"integer"}

window = default_query_window(DAYS_BACK)
site_root = getattr(config, "SITE_URL", "https://www.lifeline.org.au/")
page_url = urljoin(site_root.rstrip("/") + "/", PAGE_PATH.lstrip("/"))

print(f"Project: {config.PROJECT_ID}")
print(f"Search Console dataset: {config.SEARCHCONSOLE_DATASET}")
print(f"Window: {window.start_date} to {window.end_date}")
print(f"Page path: {PAGE_PATH}")
print(f"Page URL seed: {page_url}")
print(f"Market: {COUNTRY} / {LANGUAGE}")


In [ ]:
# Existing query drivers for the selected page
query_sql = load_sql_template(
    "sql/notebooks/seo_query_drivers_by_page.sql",
    project_id=config.PROJECT_ID,
    searchconsole_dataset=config.SEARCHCONSOLE_DATASET,
)

params = build_date_params(window) + [
    bigquery.ScalarQueryParameter("page_path", "STRING", PAGE_PATH),
]

df_page_queries = run_query(client, query_sql, params=params)
if df_page_queries.empty:
    raise ValueError(f"No query data returned for page path {PAGE_PATH!r} in the selected window.")

df_existing_queries = (
    df_page_queries.groupby("query", as_index=False)
    .agg(
        clicks=("clicks", "sum"),
        impressions=("impressions", "sum"),
        avg_position=("avg_position", "mean"),
    )
    .sort_values(["clicks", "impressions"], ascending=[False, False])
)
df_existing_queries["click_share"] = df_existing_queries["clicks"] / max(float(df_existing_queries["clicks"].sum()), 1.0)
seed_queries = df_existing_queries["query"].head(TOP_EXISTING_QUERIES).tolist()

display(df_existing_queries.head(25))


In [ ]:
# Related keyword ideas from Google Ads
google_ads_status = "Google Ads keyword ideas loaded successfully."
df_keyword_ideas = pd.DataFrame()

try:
    df_keyword_ideas = generate_keyword_ideas(
        keyword_texts=seed_queries,
        page_url=page_url,
        location_name=COUNTRY,
        language_name=LANGUAGE,
        max_ideas=MAX_IDEAS,
    )
    df_keyword_ideas = df_keyword_ideas[
        df_keyword_ideas["avg_monthly_searches"].fillna(0) >= MIN_AVG_MONTHLY_SEARCHES
    ].reset_index(drop=True)
except GoogleAdsConfigurationError as exc:
    google_ads_status = (
        "Google Ads keyword ideas were skipped because credentials or package setup is incomplete. "
        f"Details: {exc}"
    )
except Exception as exc:
    google_ads_status = f"Google Ads keyword ideas failed: {exc}"

print(google_ads_status)
if not df_keyword_ideas.empty:
    display(df_keyword_ideas.head(25))


In [ ]:
# Merge external demand with current page coverage
if df_keyword_ideas.empty:
    df_page_opportunities = pd.DataFrame()
    print("No external keyword ideas available yet. Existing query drivers above are still usable for internal review.")
else:
    df_page_opportunities = annotate_page_opportunities(
        df_keyword_ideas,
        df_existing_queries[["query", "clicks", "impressions", "avg_position"]],
    )
    recommendation_columns = [
        "keyword",
        "avg_monthly_searches",
        "current_impressions",
        "current_clicks",
        "current_avg_position",
        "lexical_similarity",
        "opportunity_category",
        "recommended_action",
        "opportunity_score",
    ]
    display(df_page_opportunities[recommendation_columns].head(30))


In [ ]:
# Keyword opportunity scatter
if df_page_opportunities.empty:
    print("Scatter chart skipped because no external keyword ideas were returned.")
else:
    df_chart = df_page_opportunities.copy()
    df_chart["current_impressions_chart"] = df_chart["current_impressions"].fillna(0).clip(lower=1)
    df_chart["avg_monthly_searches_chart"] = df_chart["avg_monthly_searches"].fillna(0).clip(lower=1)
    fig = px.scatter(
        df_chart,
        x="current_impressions_chart",
        y="avg_monthly_searches_chart",
        color="opportunity_category",
        size="opportunity_score",
        hover_name="keyword",
        hover_data={
            "current_clicks": True,
            "current_avg_position": ':.2f',
            "lexical_similarity": ':.2f',
            "recommended_action": True,
            "opportunity_score": ':.1f',
        },
        template="lifeline",
        title=f"Keyword Expansion Opportunities for {PAGE_PATH} ({window.start_date} to {window.end_date})",
        labels={
            "current_impressions_chart": "Current GSC impressions for exact query",
            "avg_monthly_searches_chart": "External avg monthly searches",
        },
    )
    fig.update_layout(height=650)
    fig.update_xaxes(type="log", rangemode="tozero")
    fig.update_yaxes(type="log", rangemode="tozero")
    lifeline_theme.add_lifeline_logo(fig)
    fig.show()


In [ ]:
# Best terms to expand on-page and split into standalone content
if df_page_opportunities.empty:
    print("No opportunity tables yet because external keyword ideas were not available.")
else:
    df_expand_on_page = df_page_opportunities[
        df_page_opportunities["recommended_action"] == "expand_on_page"
    ][[
        "keyword",
        "avg_monthly_searches",
        "current_impressions",
        "current_avg_position",
        "opportunity_category",
        "opportunity_score",
    ]].head(20)

    df_new_article = df_page_opportunities[
        df_page_opportunities["recommended_action"] == "consider_new_article"
    ][[
        "keyword",
        "avg_monthly_searches",
        "lexical_similarity",
        "competition",
        "competition_index",
        "opportunity_score",
    ]].head(20)

    print("Best terms to expand on the current page")
    display(df_expand_on_page)

    print("Best terms to consider as standalone content")
    display(df_new_article)


In [ ]:
# Final editable recommendation table for editorial action
if df_page_opportunities.empty:
    print("The final recommendation table will populate once Google Ads keyword ideas are available.")
else:
    df_recommendations = df_page_opportunities[[
        "keyword",
        "avg_monthly_searches",
        "current_query",
        "current_impressions",
        "current_clicks",
        "current_avg_position",
        "lexical_similarity",
        "opportunity_category",
        "recommended_action",
        "opportunity_score",
    ]].copy()
    df_recommendations["editorial_notes"] = ""
    df_recommendations["decision"] = ""
    display(df_recommendations.head(50))
